# Books & Interactions Processing Pipeline: Fantasy & Paranormal

This notebook implements a self-contained data pipeline for the Goodreads Fantasy category. Every step is grounded in findings from [EDA_Fantasy.ipynb](EDA_Fantasy.ipynb) with exact statistics cited for justification.

## Section 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gzip
import json
import re
import html
import sys
import os

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Add src to path if not already there
sys.path.append('..')
from src.config import CATEGORIES
from src.utils.io import read_parquet_chunks
from src.utils.cleaning import empty_strings_to_na, normalize_review_text

CATEGORY = 'fantasy_paranormal'
cfg = CATEGORIES[CATEGORY]
BOOKS_IN_PATH = cfg.interim_dir / 'books_reduced.parquet'
INTERACTIONS_IN_PATH = cfg.interim_dir / 'interactions_reduced.parquet'
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(cfg.display_name)
print(BOOKS_IN_PATH)
print(INTERACTIONS_IN_PATH)
print(OUTPUT_DIR)


Fantasy & Paranormal
/home/nakato/projects/BigBook/data/interim/fantasy_paranormal/books_reduced.parquet
/home/nakato/projects/BigBook/data/interim/fantasy_paranormal/interactions_reduced.parquet
/home/nakato/projects/BigBook/data/processed/fantasy_paranormal


## Section 1 — Inline Helper Functions

In [2]:
def parse_goodreads_dates(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    return df

def clean_text(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_authors(authors_list):
    if not isinstance(authors_list, list) or len(authors_list) == 0:
        return pd.Series({'primary_author_id': np.nan, 'author_count': 0})
    primary = [a['author_id'] for a in authors_list if a.get('role', '') == '']
    return pd.Series({
        'primary_author_id': primary[0] if primary else authors_list[0]['author_id'],
        'author_count': len(authors_list)
    })

def extract_series(series_list):
    if not isinstance(series_list, list):
        return pd.Series({'series_count': 0})
    return pd.Series({'series_count': len(series_list)})

def extract_shelves(shelves_list):
    if not isinstance(shelves_list, list) or len(shelves_list) == 0:
        return pd.Series({'top_shelf': np.nan, 'to_read_count': 0})
    to_read = [int(s['count']) for s in shelves_list if 'to-read' in s['name'].lower()]
    return pd.Series({
        'top_shelf': shelves_list[0]['name'],
        'to_read_count': sum(to_read) if to_read else 0
    })

def encode_engagement(row):
    if row['is_read'] and not pd.isna(row['rating_clean']): return 'rated'
    if not pd.isna(row['review_text_incomplete']) and len(row['review_text_incomplete']) > 10: return 'reviewed'
    if row['is_read']: return 'read_no_rating'
    return 'shelf_only'

def cap_at_p99(series):
    p99 = series.quantile(0.99)
    return series.clip(upper=p99)

def empty_strings_to_na(df):
    return df.replace(r'^\s*$', np.nan, regex=True)

def read_jsonl_sample(path, nrows=10000):
    data = []
    with gzip.open(path, 'rt') as f:
        for i, line in enumerate(f):
            if i >= nrows: break
            data.append(json.loads(line))
    return pd.DataFrame(data)

## Section 2 — Books Pipeline

In [3]:
books = pd.read_parquet(BOOKS_IN_PATH)
print(f"Loaded {len(books)} reduced books")


Loaded 31281 reduced books


In [4]:
books = empty_strings_to_na(books)

### 2.1 Empty strings → NA
**EDA N°3 finding**: all missing values are encoded as empty strings. Specific sparsity: `edition_information` 89.7%, `asin` 72.6%, `kindle_asin` 51.6%, `isbn` 49.9% (Fantasy). No true `NaN` exists in the raw JSON.

### 2.2 Numeric columns
**EDA 3 finding**: numeric fields (`ratings_count`, `text_reviews_count`, `num_pages`, `publication_year`, `publication_month`, `publication_day`) arrive as dtype `str` in the raw JSON.

In [19]:
num_cols = ['ratings_count', 'text_reviews_count', 'num_pages', 'publication_year', 'publication_month', 'publication_day']
for col in num_cols:
    if col in books.columns:
        books[col] = pd.to_numeric(books[col], errors='coerce')

### 2.3 `is_ebook` boolean
**EDA 3 finding**: field is a raw string `"true"/"false"`. Converted to boolean for consistent typing and retained for format-based segmentation.

In [20]:
books['is_ebook'] = books['is_ebook'].map({'true': True, 'false': False})

### 2.4 Author extraction + role filter
**EDA 4 finding**: of all author entries in Fantasy, 48,987 roles are empty string (`role=""`) — these are primary authors. Non-empty roles include Editor (598), Pseudonym (36), etc. Using entries without a role filter contaminates `primary_author_id`.

In [ ]:
author_features = books['authors'].apply(extract_authors)
books = books.drop(columns=[c for c in author_features.columns if c in books.columns])
books = pd.concat([books, author_features], axis=1)

### 2.5 Series extraction + `is_in_series`
**EDA 6 Fantasy finding**: 72.5% of books are in ≥1 series. Series books have an average rating of 3.989 vs 3.822 for standalones (+0.167), indicating strong survivorship bias.

In [23]:
series_features = books['series'].apply(extract_series)
books = books.drop(columns=[c for c in series_features.columns if c in books.columns])
books = pd.concat([books, series_features], axis=1)
books['is_in_series'] = books['series_count'] > 0

### 2.6 Shelves + `to_read_count`
**EDA 5 finding**: `to_read_count` is a demand/hype signal with r≈0.09 correlation with `average_rating` (weak positive). Fantasy scale: mean=6,202, median=388, max=543,151. It must be exposed separately from genre shelves.

In [24]:
shelf_features = books['popular_shelves'].apply(extract_shelves)
books = books.drop(columns=[c for c in shelf_features.columns if c in books.columns])
books = pd.concat([books, shelf_features], axis=1)

### 2.7 `publication_year_clean`
**EDA 10 Fantasy finding**: `publication_year` range is 12–29,017; 20 records before 1450 and 8 after 2026 are explicit data entry errors. Clean version clips to [1450, 2026].

In [25]:
books['publication_year_clean'] = np.where(
    books['publication_year'].between(1450, 2026),
    books['publication_year'],
    pd.NA
)

### 2.9 Dedup by `book_id`
**EDA 11 finding**: 0 duplicate `book_id` confirmed, but 13,430 duplicate `work_id` (different editions). Dedup on `book_id` is a safety step.

In [26]:
books = books.drop_duplicates('book_id', keep='first')

## Section 3 — Interactions Pipeline

### 3.1 Load
Large file (2.7 GB compressed). Using chunked iterator to avoid OOM.

In [27]:
chunks = []
chunk_iter = read_parquet_chunks(INTERACTIONS_IN_PATH, chunksize=200_000)

total_processed = 0

for chunk in chunk_iter:
    chunk = empty_strings_to_na(chunk)
    chunk = parse_goodreads_dates(chunk, ['date_added', 'date_updated', 'read_at', 'started_at'])
    chunk['rating_missing'] = (chunk['rating'] == 0)
    chunk['rating_clean'] = chunk['rating'].where(chunk['rating'].between(1, 5))
    text_col = 'review_text' if 'review_text' in chunk.columns else 'review_text_incomplete'
    chunk['review_text_clean'] = chunk[text_col].apply(normalize_review_text)
    chunk['has_review_text'] = chunk['review_text_clean'].notna()
    chunk['reading_duration_days'] = (chunk['read_at'] - chunk['started_at']).dt.days
    chunk.loc[chunk['reading_duration_days'] < 0, 'reading_duration_days'] = np.nan
    chunk['has_reading_duration'] = chunk['reading_duration_days'].notna()
    
    def get_mode(row):
        if row['is_read'] == False and row['rating_missing'] and not row['has_review_text']:
            return 'shelf_only'
        if pd.notna(row['rating_clean']) and row['has_review_text']:
            return 'reviewed'
        if pd.notna(row['rating_clean']):
            return 'rated'
        return 'read_no_rating'
    
    chunk['engagement_mode'] = chunk.apply(get_mode, axis=1)
    
    chunks.append(chunk)
    total_processed += len(chunk)

interactions = pd.concat(chunks, ignore_index=True)
print(f"Processed {len(interactions)} reduced interactions")


/tmp/ipykernel_590439/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_590439/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_590439/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
/tmp/ipykernel_590439/2474656542.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `date

Processed 21575033 reduced interactions


### 3.2 Dedup by `review_id`
**EDA 11 finding**: 0 duplicate `review_id` confirmed. Strategy: keep row with latest `date_updated` as a precaution.

In [28]:
interactions = interactions.sort_values('date_updated').drop_duplicates('review_id', keep='last')

## Section 4 — User-Level Aggregates
**Justification (EDA 8.1 Fantasy)**: 2,112 users with ≥1 rating; 53.8% generous (mean ≥ 3.67). Collaborative filtering requires `user_rating_bias` adjustment.

In [29]:
global_mean = interactions['rating_clean'].mean()
user_aggs = interactions[interactions['rating_clean'].notna()].groupby('user_id').agg(
    user_mean_rating=('rating_clean', 'mean'),
    user_rating_std=('rating_clean', 'std'),
    user_rating_count=('rating_clean', 'count')
).reset_index()

user_aggs['user_rating_bias'] = user_aggs['user_mean_rating'] - global_mean
interactions = interactions.merge(user_aggs, on='user_id', how='left')

## Section 5 — Book-Level Aggregates
**Justification (EDA 9)**: 78.5% of books have <1 interaction; 98.3% have <10. The `is_cold_start` flag is essential for routing.

In [30]:
books['book_id'] = books['book_id'].astype(str)
interactions['book_id'] = interactions['book_id'].astype(str)

book_aggs = interactions.groupby('book_id').agg(
    interaction_count=('book_id', 'count'),
    mean_rating=('rating_clean', 'mean')
).reset_index()

book_aggs['is_cold_start'] = book_aggs['interaction_count'] < 10

# Drop before merge to prevent _x/_y suffixes on re-runs
if 'is_cold_start' in books.columns:
    books = books.drop(columns=['is_cold_start', 'interaction_count', 'mean_rating'], errors='ignore')

books = books.merge(book_aggs, on='book_id', how='left')
books['is_cold_start'] = books['is_cold_start'].fillna(True).astype(bool)

if 'user_rating_bias' in interactions.columns:
    interactions = interactions.drop(columns=['user_rating_bias', 'user_mean_rating', 'user_rating_std', 'user_rating_count'], errors='ignore')

interactions = interactions.merge(user_aggs[['user_id', 'user_rating_bias']], on='user_id', how='left')
interactions['user_rating_bias'] = interactions['user_rating_bias'].fillna(0)

## Section 6 — Outlier Treatment
**Justification (EDA 10 Fantasy)**: `ratings_count` has 7,215 IQR outliers (max 534,960). P99-capped versions are added for modeling.

In [34]:
for col in ['ratings_count', 'text_reviews_count', 'num_pages', 'interaction_count']:
    if col in books.columns:
        books[f'{col}_p99'] = cap_at_p99(books[col].fillna(0))

## Section 7 — Feature Summary
Visual diagnostic section to inspect available features, data types, non-null counts, and descriptive statistics before export.

In [35]:
from IPython.display import display

def _safe_hashable_value(x):
    try:
        hash(x)
        return x
    except TypeError:
        return str(x)

def _safe_nunique(series):
    try:
        return series.nunique(dropna=True)
    except TypeError:
        return series.map(_safe_hashable_value).nunique(dropna=True)

def _safe_describe(df):
    df_safe = df.copy()
    for col in df_safe.columns:
        try:
            _ = df_safe[col].nunique(dropna=True)
        except TypeError:
            df_safe[col] = df_safe[col].map(_safe_hashable_value)
    return df_safe.describe(include='all').T

def feature_overview(df, name):
    summary = pd.DataFrame({
        'feature': df.columns,
        'dtype': df.dtypes.astype(str).values,
        'non_null_count': df.notna().sum().values,
        'null_count': df.isna().sum().values,
        'unique_count': [_safe_nunique(df[col]) for col in df.columns]
    }).sort_values(['non_null_count', 'feature'], ascending=[False, True]).reset_index(drop=True)

    print(f"\n{name} — feature overview")
    display(summary)

    print(f"\n{name} — describe(include='all').T")
    display(_safe_describe(df))

feature_overview(books, 'books')
feature_overview(interactions, 'interactions')


books — feature overview


,feature,dtype,non_null_count,null_count,unique_count
0,author_count,float64,31281,0,author_count 1 author_count 1 dtype: int64
1,author_count,float64,31281,0,author_count 1 author_count 1 dtype: int64
2,authors,object,31281,0,8101
3,average_rating,float64,31281,0,206
4,book_id,str,31281,0,31281
5,interaction_count,int64,31281,0,3198
6,interaction_count_p99,float64,31281,0,2887
7,is_cold_start,bool,31281,0,1
8,is_in_series,bool,31281,0,1
9,language_code,str,31281,0,4



books — describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
text_reviews_count,31281.0,NaN,NaN,NaN,311.185064,1495.019946,1.0,40.0,78.0,176.0,90766.0
series,31281,20771,[],3810,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language_code,31281,4,eng,23012,NaN,NaN,NaN,NaN,NaN,NaN,NaN
popular_shelves,31281,26723,"[{'count': '525817', 'name': 'to-read'}\n {'co...",23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_ebook,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
average_rating,31281.0,NaN,NaN,NaN,3.985126,0.284377,1.28,3.8,4.0,4.19,4.82
description,29976,27467,Bilbo Baggins is a reasonably typical hobbit: ...,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
format,24024,60,Paperback,9461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
authors,31281,8101,"[{'author_id': '1654', 'role': ''}]",207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
publisher,23950,3737,VIZ Media LLC,441,NaN,NaN,NaN,NaN,NaN,NaN,NaN



interactions — feature overview


,feature,dtype,non_null_count,null_count,unique_count
0,book_id,str,21575033,0,31281
1,date_added,"datetime64[us, UTC]",21575033,0,20371505
2,date_updated,"datetime64[us, UTC]",21575033,0,20149577
3,engagement_mode,str,21575033,0,2
4,has_reading_duration,bool,21575033,0,2
5,has_review_text,bool,21575033,0,2
6,is_read,bool,21575033,0,1
7,rating,float64,21575033,0,5
8,rating_clean,float64,21575033,0,5
9,rating_missing,bool,21575033,0,1



interactions — describe(include='all').T


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
user_id,21575033,459287,98f7558f89bf885fd88ff12cc0b8a705,2701,NaN,NaN,NaN,NaN,NaN,NaN,NaN
book_id,21575033,31281,3,252625,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_id,21575033,21575033,5e9e5e0436ff546058465f1848118fd5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_read,21575033,1,True,21575033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,21575033.0,NaN,NaN,NaN,4.035001,1.0,3.0,4.0,5.0,5.0,0.975253
review_text_incomplete,2387075,2302438,<br /><br />,3270,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,21575033,NaN,NaN,NaN,2013-09-14 21:47:28.575195+00:00,1992-01-01 08:00:00+00:00,2012-03-27 17:30:28+00:00,2013-09-11 07:44:41+00:00,2015-06-14 14:27:14+00:00,2017-11-04 06:34:48+00:00,NaN
date_updated,21575033,NaN,NaN,NaN,2014-01-29 22:29:30.684602+00:00,2006-08-29 18:00:42+00:00,2012-07-10 01:51:22+00:00,2014-03-02 05:33:02+00:00,2015-12-20 04:15:45+00:00,2017-11-05 20:21:30+00:00,NaN
read_at,8003817,NaN,NaN,NaN,2014-04-08 09:20:24.395224+00:00,0101-01-02 00:00:00+00:00,2012-12-11 08:00:00+00:00,2014-11-16 08:00:00+00:00,2016-05-01 07:00:00+00:00,4732-01-25 08:00:00+00:00,NaN
started_at,5650866,NaN,NaN,NaN,2014-12-07 03:04:32.520559+00:00,1910-01-01 08:00:00+00:00,2013-06-25 07:00:00+00:00,2015-03-25 20:48:49.500000+00:00,2016-07-22 07:00:00+00:00,4732-01-25 08:00:00+00:00,NaN


## Section 8 — Output

In [36]:
books = books.loc[:, ~books.columns.duplicated(keep='last')]
interactions = interactions.loc[:, ~interactions.columns.duplicated(keep='last')]

books.to_parquet(OUTPUT_DIR / 'books_curated.parquet', index=False)
interactions.to_parquet(OUTPUT_DIR / 'interactions_curated.parquet', index=False)
print(f"Saved curated files to {OUTPUT_DIR}")

Saved curated files to /home/nakato/projects/BigBook/data/processed/fantasy_paranormal


## Section 9 — Validation

In [37]:
# Original assertions
assert books['book_id'].notna().all()
assert interactions['book_id'].notna().all()
assert interactions['rating_clean'].dropna().between(1, 5).all()
assert books['book_id'].duplicated().sum() == 0
assert interactions['review_id'].dropna().duplicated().sum() == 0

# New (EDA-grounded) assertions
assert 'authors' in books.columns
assert 'popular_shelves' in books.columns
assert books['publication_year_clean'].dropna().between(1450, 2026).all()
valid_modes = {'shelf_only', 'rated', 'reviewed', 'read_no_rating'}
assert interactions['engagement_mode'].isin(valid_modes).all()
assert interactions['reading_duration_days'].dropna().ge(0).all()
assert books['is_cold_start'].dtype == bool or books['is_cold_start'].dtype == 'bool'
assert interactions['user_rating_bias'].notna().sum() > 0

print("All validations passed!")

All validations passed!
